# Seat Changes without GRCs

What if GRC seats had been split proportionally based on votes? Set `SCENARIO` in the next cell, then **Run All** to get the figures and tables for that scenario.

| Scenario | Rounding rule for fractional seats |
|---|---|
| `ProPAP` | PAP gets the rounded-up seat split |
| `AntiPAP` | PAP gets the rounded-down seat split |
| `ProWinner` | The actual winner of the GRC gets the rounded-up seat split |

Results are written to `<SCENARIO>_results/`.

In [ ]:
# Choose scenario: 'ProPAP', 'AntiPAP' or 'ProWinner'
SCENARIO = 'ProPAP'


scenario_titles = {
    'ProPAP': 'Pro-PAP',
    'AntiPAP': 'Anti-PAP',
    'ProWinner': 'Pro-Winner',
}
assert SCENARIO in scenario_titles, "SCENARIO must be 'ProPAP', 'AntiPAP' or 'ProWinner'"
RESULTS_DIR = SCENARIO + '_results'

In [ ]:
import pandas as pd
import math
import os

In [ ]:
colors = {
    "People\'s Action Party": '#0000FF',  # Blue
    "Workers' Party": '#FF0000',  # Red
    "National Solidarity Party": '#00FF00',  # Green
    "Singapore Democratic Party": '#FFFF00',  # Yellow
    "Reform Party": '#FFA500',  # Orange
    "Singapore People's Party": '#800080',  # Purple
    "Progress Singapore Party": '#00FFFF',  # Cyan
    "People's Power Party": '#FF00FF',  # Magenta
    "Red Dot United": '#A52A2A',  # Brown
    "Singapore Democratic Alliance": '#808080',  # Gray
    "Singaporeans First": '#8B4513',  # SaddleBrown
    "Peoples Voice": '#FFD700',  # Gold
    "United People's Front": '#000000', # Black
    "Singapore Justice Party": '#FFC0CB',  # Pink
    "People's Alliance for Reform": '#FF7F50',  # Coral
    "Singapore United Party": '#4682B4',  # SteelBlue
    "Independent": '#3b3b3b'  # Gray

}

In [ ]:
def allocate_seats(party, vote_percentage, total_seats):
    """Split a GRC's seats proportionally based on votes;
    the rounding of fractional seats depends on the scenario."""

    if SCENARIO == 'ProPAP':
        round_up = (party == "People's Action Party")
    elif SCENARIO == 'AntiPAP':
        round_up = (party != "People's Action Party")
    else:  # ProWinner: the actual winner gets the rounded-up seat split
        round_up = (vote_percentage > 50)

    if round_up:
        return math.ceil(total_seats * vote_percentage / 100)
    else:
        return math.floor(total_seats * vote_percentage / 100)

In [ ]:
GE_years = [1988,1991,1997,2001,2006,2011,2015,2020,2025]

os.makedirs(RESULTS_DIR, exist_ok=True)

all_seat_changes=pd.DataFrame(columns = ['Party','Seat Change','Year'])


for year in GE_years:

    ## Set up dataframe
    file_path = 'Actual_results/'+ str(year)+'-Table 1.csv'
    df = pd.read_csv(file_path)

    # Remove empty rows and columns
    df = df.dropna(how='all').dropna(axis=1, how='all')

    df.columns = df.iloc[0]
    df = df[1:]

    # Reset the index
    df.reset_index(drop=True, inplace=True)


    ## Data Cleaning

    # Fill up rows, which dont have Constituency and Seats, with the respective constituency and seats
    for i in range(1, len(df)):
        if pd.isna(df.at[i, df.columns[0]]):

            df.iloc[i][0] = df.iloc[i-1][0]
            df.iloc[i][1] = df.iloc[i-1][1]

    # Remove SMCs
    df = df[df['Seats'] != '1']

    # Remove walkovers (if no vote percentage)
    df = df.dropna(subset='%')

    df['%']=df['%'].str[:5]
    df['%']=df['%'].astype(float)
    df['Seats']=df['Seats'].astype(int)


    # Remove bug substring
    df['Party']=df['Party'].str.replace('\xa0','',regex=False)

    df.reset_index(drop=True, inplace=True)

    ## Calculations

    # Calculate Old Seats
    df['Old Seats']=0
    for i in range(0, len(df)):
        total_seats = df.iloc[i]['Seats']
        vote_percentage=df.iloc[i]['%']
        if df.iloc[i]['%']> 50:
            df.at[i,'Old Seats'] = total_seats
        else:
            df.at[i,'Old Seats'] = 0

    # Calculate new seats (scenario-dependent)
    df['New Seats']=0
    for i in range(0, len(df)):
        df.at[i,'New Seats'] = allocate_seats(df.iloc[i]['Party'], df.iloc[i]['%'], df.iloc[i]['Seats'])

    ## Calculate Seat Changes
    df['Seat Change']=0
    for i in range(0, len(df)):
        df.at[i,'Seat Change'] = df.iloc[i]['New Seats']-df.iloc[i]['Old Seats']

    df.to_csv(RESULTS_DIR+'/'+str(year)+'_results.csv',index=False)

    ## Collate Seat Changes
    seat_changes = df.groupby('Party')['Seat Change'].sum().reset_index()
    seat_changes['Year']=year

    all_seat_changes=pd.concat([all_seat_changes,seat_changes],ignore_index=True)

all_seat_changes

In [ ]:
all_seat_changes.to_csv(RESULTS_DIR+'/all_results.csv',index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = all_seat_changes

# Group all parties except People's Action Party and Workers' Party under "Others"
data['Party Grouped'] = data['Party'].apply(lambda x: x if x in ["People's Action Party", "Workers' Party", "Singapore Democratic Party", "Singapore People's Party"] else 'Others')

# Summarize the seat changes by the new grouping
grouped_summary = data.groupby(['Year', 'Party Grouped'])['Seat Change'].sum().reset_index()

# Adjust the grouped bar chart to ensure matching data lengths
plt.figure(figsize=(8, 6))

# Define the width of each bar
bar_width = 0.10

# Filter data for each group
pap_data = grouped_summary[grouped_summary['Party Grouped'] == "People's Action Party"]
wp_data = grouped_summary[grouped_summary['Party Grouped'] == "Workers' Party"]
sdp_data = grouped_summary[grouped_summary['Party Grouped'] == "Singapore Democratic Party"]
spp_data = grouped_summary[grouped_summary['Party Grouped'] == "Singapore People's Party"]
others_data = grouped_summary[grouped_summary['Party Grouped'] == 'Others']

# Generate positions for the bars
years = sorted(grouped_summary['Year'].unique())
positions = {year: index for index, year in enumerate(years)}

# Adjust positions for each group
pap_positions = [positions[year] - 2*bar_width for year in pap_data['Year']]
wp_positions = [positions[year] - bar_width for year in wp_data['Year']]
sdp_positions = [positions[year] for year in sdp_data['Year']]
spp_positions = [positions[year] + bar_width for year in spp_data['Year']]
others_positions = [positions[year] + 2*bar_width for year in others_data['Year']]

# Plotting bars for each party group with specified colors
plt.bar(pap_positions, pap_data['Seat Change'], color=colors["People\'s Action Party"], width=bar_width, label="People's Action Party")
plt.bar(wp_positions, wp_data['Seat Change'], color=colors["Workers' Party"], width=bar_width, label="Workers' Party")
plt.bar(sdp_positions, sdp_data['Seat Change'], color=colors["Singapore Democratic Party"], width=bar_width, label= "Singapore Democratic Party")
plt.bar(spp_positions, spp_data['Seat Change'], color=colors["Singapore People's Party"], width=bar_width, label= "Singapore People's Party")
plt.bar(others_positions, others_data['Seat Change'], color='green', width=bar_width, label='Others')

# Set the x-axis to show only the election years
plt.title('Seat Changes without GRCs ('+scenario_titles[SCENARIO]+')')
plt.xlabel('GE Year')
plt.ylabel('Seat Change')
plt.xticks(range(len(years)), years)  # Set positions for x-axis labels
plt.legend()
plt.grid(False)
plt.axhline(y=0, color='black', linewidth=0.8)  # Add the y=0 horizontal line
plt.show()

In [ ]:
## Look at specific year's results

year = 2025

df = pd.read_csv(RESULTS_DIR+'/'+str(year)+'_results.csv')

df

In [ ]:
# Look at specific year's seat change for all parties

df = all_seat_changes
year = 2025

df = df[df['Year'] == year]

df